# 🩸 Blood Glucose Prediction Pipeline (Memory Optimized)

**Task**: Predict CGM 30 minutes into the future (6 steps × 5-min intervals)  
**Model**: XGBoost Regressor with GPU acceleration  
**Objective**: `reg:squarederror`

---
⚠️ **MEMORY OPTIMIZED** - Uses Polars Lazy API + Streaming for 132M+ rows  
⚠️ **GPU OPTIMIZED** - Uses DeviceQuantileDMatrix for 111M+ row training  
⚠️ **DISTINCT FROM INSULIN MODEL** - Uses separate naming convention

## Cell 1: Setup & Configuration

In [ ]:
# Install required packages
!pip install -q polars xgboost matplotlib

# Imports
import polars as pl
import xgboost as xgb
import numpy as np
import matplotlib.pyplot as plt
import json
import os
import gc
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✅ Setup complete!")
print(f"Polars version: {pl.__version__}")
print(f"XGBoost version: {xgb.__version__}")

In [ ]:
# ============================================================
# CONFIGURATION - DISTINCT PATHS FOR GLUCOSE PREDICTION
# ============================================================

BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data"

# Raw data paths
RAW_TRAIN = f"{BASE_PATH}/raw/metabonet_public_train.parquet"
RAW_TEST = f"{BASE_PATH}/raw/metabonet_public_test.parquet"

# Processed data paths (DISTINCT from insulin model)
PROC_TRAIN = f"{BASE_PATH}/processed/train_ready_for_glucose_prediction.parquet"
PROC_TEST = f"{BASE_PATH}/processed/test_ready_for_glucose_prediction.parquet"

# Model output (DISTINCT from insulin model)
MODEL_OUTPUT = f"{BASE_PATH}/processed/xgboost_glucose_model.json"

# Columns to keep from raw data (EARLY PROJECTION - Critical for memory)
RAW_KEEP_COLS = ['id', 'date', 'CGM', 'bolus', 'carbs', 'basal', 'insulin_type_bolus', 'gender', 'age', 'weight']

# Feature configuration (13 features)
FEATURE_COLS = [
    'CGM', 'bolus', 'carbs', 'basal',
    'time_sin', 'time_cos', 'is_weekend',
    'cgm_delta_5min', 'cgm_delta_15min',
    'age', 'weight',
    'gender_encoded', 'insulin_type_encoded'
]

TARGET_COL = 'CGM_future'
FORECAST_STEPS = 6  # 6 steps × 5-min = 30 min forecast

# Ensure directories exist
os.makedirs(f"{BASE_PATH}/processed", exist_ok=True)

print("📁 Configuration:")
print(f"  RAW_TRAIN:   {RAW_TRAIN}")
print(f"  RAW_TEST:    {RAW_TEST}")
print(f"  PROC_TRAIN:  {PROC_TRAIN}")
print(f"  PROC_TEST:   {PROC_TEST}")
print(f"  MODEL:       {MODEL_OUTPUT}")
print(f"\n🔧 Early Projection Columns: {RAW_KEEP_COLS}")
print(f"🎯 Target: {TARGET_COL} (CGM shifted by -{FORECAST_STEPS} rows = 30 min ahead)")
print(f"📊 Final Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")

## Step 1: Preprocess TRAIN Data (Memory Optimized - Streaming)

In [ ]:
print("="*70)
print("📥 STEP 1: PREPROCESS TRAIN DATA (STREAMING MODE)")
print("="*70)
print("⚡ Using Polars Lazy API + Streaming to handle 132M+ rows")
print("⚡ Data will be processed in chunks and streamed directly to disk")
print()

# ============================================================
# 1. SCAN (Lazy Load) - No data loaded into RAM yet
# ============================================================
print("📂 Step 1.1: Scanning parquet (lazy)...")
q = pl.scan_parquet(RAW_TRAIN)
print(f"   Schema: {q.collect_schema().names()}")

# ============================================================
# 2. EARLY PROJECTION - Drop unnecessary columns immediately
# ============================================================
print("\n🔧 Step 1.2: Early projection (keeping only required columns)...")

# Get available columns from schema
available_cols = q.collect_schema().names()
keep_cols = [c for c in RAW_KEEP_COLS if c in available_cols]
missing_cols = [c for c in RAW_KEEP_COLS if c not in available_cols]

print(f"   Keeping: {keep_cols}")
if missing_cols:
    print(f"   Missing (will be filled): {missing_cols}")

# Select only required columns
q = q.select(keep_cols)

# ============================================================
# 3. SORT BY PATIENT AND TIME (Critical for time series)
# ============================================================
print("\n🔧 Step 1.3: Sorting by ['id', 'date']...")
q = q.sort(['id', 'date'])

# ============================================================
# 4. FILTER NULL CGM VALUES
# ============================================================
print("\n🔧 Step 1.4: Filtering null CGM values...")
q = q.filter(pl.col('CGM').is_not_null())

# ============================================================
# 5. FILL NULLS FOR NUMERIC COLUMNS
# ============================================================
print("\n🔧 Step 1.5: Filling null values...")
q = q.with_columns([
    pl.col('bolus').fill_null(0.0).cast(pl.Float32),
    pl.col('carbs').fill_null(0.0).cast(pl.Float32),
    pl.col('basal').fill_null(0.0).cast(pl.Float32),
    pl.col('age').fill_null(-1.0).cast(pl.Float32),
    pl.col('weight').fill_null(-1.0).cast(pl.Float32),
    pl.col('CGM').cast(pl.Float32),
])

# ============================================================
# 6. TIME FEATURES
# ============================================================
print("\n🔧 Step 1.6: Creating time features...")
q = q.with_columns([
    pl.col('date').dt.hour().alias('hour'),
    pl.col('date').dt.weekday().alias('weekday'),
])

q = q.with_columns([
    (2 * np.pi * pl.col('hour') / 24).sin().cast(pl.Float32).alias('time_sin'),
    (2 * np.pi * pl.col('hour') / 24).cos().cast(pl.Float32).alias('time_cos'),
    pl.when(pl.col('weekday') >= 5).then(1.0).otherwise(0.0).cast(pl.Float32).alias('is_weekend'),
])

# ============================================================
# 7. CGM DELTA FEATURES (Trend)
# ============================================================
print("\n🔧 Step 1.7: Creating CGM delta features...")
q = q.with_columns([
    (pl.col('CGM') - pl.col('CGM').shift(1).over('id')).cast(pl.Float32).alias('cgm_delta_5min'),
    (pl.col('CGM') - pl.col('CGM').shift(3).over('id')).cast(pl.Float32).alias('cgm_delta_15min'),
])

# ============================================================
# 8. CATEGORICAL ENCODING
# ============================================================
print("\n🔧 Step 1.8: Encoding categorical features...")

# Gender encoding
if 'gender' in keep_cols:
    q = q.with_columns(
        pl.when(pl.col('gender').str.to_lowercase() == 'male').then(1.0)
        .when(pl.col('gender').str.to_lowercase() == 'female').then(0.0)
        .otherwise(-1.0)
        .cast(pl.Float32)
        .alias('gender_encoded')
    )
else:
    q = q.with_columns(pl.lit(-1.0).cast(pl.Float32).alias('gender_encoded'))

# Insulin type encoding
if 'insulin_type_bolus' in keep_cols:
    q = q.with_columns(
        pl.when(pl.col('insulin_type_bolus').str.to_lowercase().str.contains('rapid')).then(1.0)
        .when(pl.col('insulin_type_bolus').str.to_lowercase().str.contains('long')).then(2.0)
        .when(pl.col('insulin_type_bolus').str.to_lowercase().str.contains('mix')).then(3.0)
        .otherwise(0.0)
        .cast(pl.Float32)
        .alias('insulin_type_encoded')
    )
else:
    q = q.with_columns(pl.lit(0.0).cast(pl.Float32).alias('insulin_type_encoded'))

# ============================================================
# 9. TARGET CREATION (CGM_future = CGM shifted -6)
# ============================================================
print("\n🔧 Step 1.9: Creating target (CGM_future = CGM shifted -6 rows)...")
q = q.with_columns(
    pl.col('CGM').shift(-FORECAST_STEPS).over('id').cast(pl.Float32).alias('CGM_future')
)

# Filter out null targets
q = q.filter(pl.col('CGM_future').is_not_null())

# ============================================================
# 10. FINAL SELECTION (Features + Target only)
# ============================================================
print("\n🔧 Step 1.10: Final column selection...")
final_cols = FEATURE_COLS + [TARGET_COL]
q = q.select(final_cols)

# Drop any remaining nulls from delta calculations
q = q.drop_nulls()

# ============================================================
# 11. STREAMING SAVE TO DISK (No RAM collection!)
# ============================================================
print(f"\n💾 Step 1.11: Streaming to disk: {PROC_TRAIN}")
print("   ⏳ This may take a few minutes for 132M rows...")

# SINK directly to parquet - streams in chunks, never loads full data
q.sink_parquet(PROC_TRAIN)

print("\n" + "="*70)
print("✅ STEP 1 COMPLETE - Train data preprocessed and saved!")
print("="*70)

# Verify file was created
if os.path.exists(PROC_TRAIN):
    file_size_gb = os.path.getsize(PROC_TRAIN) / (1024**3)
    print(f"\n📁 Output file: {PROC_TRAIN}")
    print(f"📊 File size: {file_size_gb:.2f} GB")
    
    # Quick row count check (lazy scan)
    row_count = pl.scan_parquet(PROC_TRAIN).select(pl.len()).collect().item()
    print(f"📊 Row count: {row_count:,}")
else:
    print("❌ Error: Output file was not created!")

## Step 2: Preprocess TEST Data (Memory Optimized - Streaming)

In [ ]:
print("="*70)
print("📥 STEP 2: PREPROCESS TEST DATA (STREAMING MODE)")
print("="*70)

# Scan test data (lazy)
print("\n📂 Scanning test parquet (lazy)...")
q_test = pl.scan_parquet(RAW_TEST)

# Get available columns
test_available = q_test.collect_schema().names()
test_keep_cols = [c for c in RAW_KEEP_COLS if c in test_available]

# Early projection
q_test = q_test.select(test_keep_cols)

# Sort
q_test = q_test.sort(['id', 'date'])

# Filter null CGM
q_test = q_test.filter(pl.col('CGM').is_not_null())

# Fill nulls
q_test = q_test.with_columns([
    pl.col('bolus').fill_null(0.0).cast(pl.Float32),
    pl.col('carbs').fill_null(0.0).cast(pl.Float32),
    pl.col('basal').fill_null(0.0).cast(pl.Float32),
    pl.col('age').fill_null(-1.0).cast(pl.Float32),
    pl.col('weight').fill_null(-1.0).cast(pl.Float32),
    pl.col('CGM').cast(pl.Float32),
])

# Time features
q_test = q_test.with_columns([
    pl.col('date').dt.hour().alias('hour'),
    pl.col('date').dt.weekday().alias('weekday'),
])

q_test = q_test.with_columns([
    (2 * np.pi * pl.col('hour') / 24).sin().cast(pl.Float32).alias('time_sin'),
    (2 * np.pi * pl.col('hour') / 24).cos().cast(pl.Float32).alias('time_cos'),
    pl.when(pl.col('weekday') >= 5).then(1.0).otherwise(0.0).cast(pl.Float32).alias('is_weekend'),
])

# CGM deltas
q_test = q_test.with_columns([
    (pl.col('CGM') - pl.col('CGM').shift(1).over('id')).cast(pl.Float32).alias('cgm_delta_5min'),
    (pl.col('CGM') - pl.col('CGM').shift(3).over('id')).cast(pl.Float32).alias('cgm_delta_15min'),
])

# Categorical encoding
if 'gender' in test_keep_cols:
    q_test = q_test.with_columns(
        pl.when(pl.col('gender').str.to_lowercase() == 'male').then(1.0)
        .when(pl.col('gender').str.to_lowercase() == 'female').then(0.0)
        .otherwise(-1.0)
        .cast(pl.Float32)
        .alias('gender_encoded')
    )
else:
    q_test = q_test.with_columns(pl.lit(-1.0).cast(pl.Float32).alias('gender_encoded'))

if 'insulin_type_bolus' in test_keep_cols:
    q_test = q_test.with_columns(
        pl.when(pl.col('insulin_type_bolus').str.to_lowercase().str.contains('rapid')).then(1.0)
        .when(pl.col('insulin_type_bolus').str.to_lowercase().str.contains('long')).then(2.0)
        .when(pl.col('insulin_type_bolus').str.to_lowercase().str.contains('mix')).then(3.0)
        .otherwise(0.0)
        .cast(pl.Float32)
        .alias('insulin_type_encoded')
    )
else:
    q_test = q_test.with_columns(pl.lit(0.0).cast(pl.Float32).alias('insulin_type_encoded'))

# Target creation
q_test = q_test.with_columns(
    pl.col('CGM').shift(-FORECAST_STEPS).over('id').cast(pl.Float32).alias('CGM_future')
)

# Filter nulls
q_test = q_test.filter(pl.col('CGM_future').is_not_null())

# Final selection
final_cols = FEATURE_COLS + [TARGET_COL]
q_test = q_test.select(final_cols)
q_test = q_test.drop_nulls()

# Stream to disk
print(f"\n💾 Streaming to disk: {PROC_TEST}")
q_test.sink_parquet(PROC_TEST)

print("\n" + "="*70)
print("✅ STEP 2 COMPLETE - Test data preprocessed and saved!")
print("="*70)

# Verify
if os.path.exists(PROC_TEST):
    file_size_gb = os.path.getsize(PROC_TEST) / (1024**3)
    row_count = pl.scan_parquet(PROC_TEST).select(pl.len()).collect().item()
    print(f"\n📁 Output file: {PROC_TEST}")
    print(f"📊 File size: {file_size_gb:.2f} GB")
    print(f"📊 Row count: {row_count:,}")

## Step 3: Model Training (GPU Optimized - DeviceQuantileDMatrix)

In [ ]:
print("="*70)
print("🚀 STEP 3: MODEL TRAINING (GPU OPTIMIZED)")
print("="*70)
print("⚡ Using DeviceQuantileDMatrix for 111M+ row GPU training")
print("⚡ This prevents GPU memory bottleneck on T4")
print()

# ============================================================
# 1. LOAD PROCESSED TRAIN DATA
# ============================================================
print("📂 Loading processed train data...")
df_train = pl.read_parquet(PROC_TRAIN)
print(f"✅ Loaded shape: {df_train.shape}")
print(f"   Memory usage: ~{df_train.estimated_size('gb'):.2f} GB")

# ============================================================
# 2. PREPARE FEATURES AND TARGET
# ============================================================
print("\n🔧 Preparing features and target...")
X_train = df_train.select(FEATURE_COLS).to_numpy().astype(np.float32)
y_train = df_train.select(TARGET_COL).to_numpy().ravel().astype(np.float32)

# Free DataFrame memory
del df_train
gc.collect()

print(f"   X_train shape: {X_train.shape}")
print(f"   y_train shape: {y_train.shape}")
print(f"   Target range: [{y_train.min():.1f}, {y_train.max():.1f}] mg/dL")
print(f"   Target mean: {y_train.mean():.1f} mg/dL")

# ============================================================
# 3. CREATE DEVICE QUANTILE DMATRIX (CRITICAL FOR LARGE DATA)
# ============================================================
print("\n⚡ Creating DeviceQuantileDMatrix (GPU-optimized)...")
print("   This quantizes data on GPU to reduce memory footprint")

dtrain = xgb.DeviceQuantileDMatrix(
    data=X_train,
    label=y_train,
    feature_names=FEATURE_COLS,
    max_bin=256  # Optimizes memory usage
)

print("✅ DeviceQuantileDMatrix created!")

# Free numpy arrays after DMatrix creation
del X_train, y_train
gc.collect()

# ============================================================
# 4. DEFINE TRAINING PARAMETERS
# ============================================================
print("\n📋 Training Parameters:")

params = {
    'objective': 'reg:squarederror',
    'tree_method': 'hist',
    'device': 'cuda',
    'sampling_method': 'gradient_based',  # Works best with DeviceQuantileDMatrix
    'max_depth': 6,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 1,
    'seed': 42,
}

num_rounds = 1000

for key, value in params.items():
    print(f"   {key}: {value}")
print(f"   num_boost_round: {num_rounds}")

# ============================================================
# 5. TRAIN MODEL
# ============================================================
print("\n⏳ Training XGBoost model...")
print("   Progress will be printed every 100 rounds")

# Custom callback for progress
def print_progress(period=100):
    def callback(env):
        if env.iteration % period == 0:
            print(f"   Round {env.iteration}: train-rmse = {env.evaluation_result_list[0][1]:.4f}")
    return callback

# Training with evaluation
evals = [(dtrain, 'train')]

model = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=num_rounds,
    evals=evals,
    verbose_eval=100,  # Print every 100 rounds
)

print("\n✅ Training complete!")

# Free DMatrix memory
del dtrain
gc.collect()

# ============================================================
# 6. SAVE MODEL
# ============================================================
print(f"\n💾 Saving model to: {MODEL_OUTPUT}")
model.save_model(MODEL_OUTPUT)
print("✅ Model saved!")

# Save feature info
feature_info_path = MODEL_OUTPUT.replace('.json', '_features.json')
feature_info = {
    "model_type": "glucose_prediction",
    "feature_names": FEATURE_COLS,
    "target": TARGET_COL,
    "forecast_horizon": "30 minutes (6 × 5-min steps)",
    "training_params": params,
    "num_boost_round": num_rounds,
    "optimization": "DeviceQuantileDMatrix with gradient_based sampling"
}

with open(feature_info_path, 'w') as f:
    json.dump(feature_info, f, indent=2)
print(f"✅ Feature info saved: {feature_info_path}")

print("\n" + "="*70)
print("✅ STEP 3 COMPLETE - Model trained and saved!")
print("="*70)

## Step 4: Evaluation

In [ ]:
print("="*70)
print("📊 STEP 4: EVALUATION")
print("="*70)

# Load processed test data
print(f"\n📂 Loading: {PROC_TEST}")
df_test = pl.read_parquet(PROC_TEST)
print(f"Test shape: {df_test.shape}")

X_test = df_test.select(FEATURE_COLS).to_numpy().astype(np.float32)
y_test = df_test.select(TARGET_COL).to_numpy().ravel().astype(np.float32)
del df_test
gc.collect()

# Create DMatrix for prediction (standard DMatrix is fine for inference)
dtest = xgb.DMatrix(data=X_test, feature_names=FEATURE_COLS)
del X_test
gc.collect()

# Predict
print("\n🔮 Making predictions...")
y_pred = model.predict(dtest)
del dtest
gc.collect()

# Metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print("\n" + "="*50)
print("🎯 GLUCOSE PREDICTION METRICS")
print("="*50)
print(f"  RMSE: {rmse:.4f} mg/dL")
print(f"  MAE:  {mae:.4f} mg/dL")
print("="*50)

# Clinical analysis
abs_errors = np.abs(y_test - y_pred)
within_15 = np.mean(abs_errors <= 15) * 100
within_20 = np.mean(abs_errors <= 20) * 100

print(f"\n🏥 Clinical Accuracy:")
print(f"  ± 15 mg/dL: {within_15:.2f}%")
print(f"  ± 20 mg/dL: {within_20:.2f}%")

In [ ]:
# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Time series
n = min(200, len(y_test))
axes[0,0].plot(range(n), y_test[:n], label='Actual', color='#2ecc71', lw=2)
axes[0,0].plot(range(n), y_pred[:n], label='Predicted', color='#e74c3c', lw=2, ls='--')
axes[0,0].set_title('Actual vs Predicted (30-min Forecast)')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# 2. Scatter (sample for performance)
sample_size = min(50000, len(y_test))
axes[0,1].scatter(y_test[:sample_size], y_pred[:sample_size], alpha=0.2, s=2)
axes[0,1].plot([0,400], [0,400], 'r--', lw=2)
axes[0,1].set_xlabel('Actual'); axes[0,1].set_ylabel('Predicted')
axes[0,1].set_title('Scatter Plot')
axes[0,1].grid(True, alpha=0.3)

# 3. Error distribution (sample)
errors = y_test[:sample_size] - y_pred[:sample_size]
axes[1,0].hist(errors, bins=100, color='#9b59b6', alpha=0.7)
axes[1,0].axvline(0, color='red', ls='--')
axes[1,0].set_title(f'Error Distribution (Mean: {np.mean(errors):.2f})')
axes[1,0].grid(True, alpha=0.3)

# 4. Feature importance
importance = model.get_score(importance_type='gain')
imp_sorted = sorted(importance.items(), key=lambda x: x[1])
features = [x[0] for x in imp_sorted]
values = [x[1] for x in imp_sorted]
axes[1,1].barh(range(len(features)), values, color='#1abc9c')
axes[1,1].set_yticks(range(len(features)))
axes[1,1].set_yticklabels(features)
axes[1,1].set_title('Feature Importance (Gain)')

plt.tight_layout()
plt.show()

print("\n🎉 PIPELINE COMPLETE!")